---
#### Web search
---

In [1]:
from openai import OpenAI

In [19]:
client = OpenAI()

In [20]:
response = client.chat.completions.create(
    model             = "gpt-4o-search-preview",  # Required for web search
    web_search_options= {},                       # Enables search
    messages          = [
        {"role": "user", "content": "Any new FDA-approved cancer treatment this week?"}
    ]
)

print(response.choices[0].message.content)

As of September 16, 2025, the U.S. Food and Drug Administration (FDA) has not announced any new cancer treatment approvals within the past week. The most recent approval occurred on September 9, 2025, when the FDA approved Johnson & Johnson's Inlexzo (also known as TAR-200) for treating high-risk non-muscle invasive bladder cancer (NMIBC) in patients unresponsive to Bacillus Calmette-Guerin (BCG) therapy and who are ineligible for or decline bladder removal surgery. ([reuters.com](https://www.reuters.com/business/healthcare-pharmaceuticals/us-fda-approves-jjs-bladder-cancer-treatment-2025-09-09/?utm_source=openai))

Inlexzo is a drug delivery system inserted into the bladder, where it remains for three weeks per cycle for up to 14 cycles, delivering sustained doses of the chemotherapy drug gemcitabine. Approval was based on a mid-stage trial demonstrating that over 82% of patients showed no cancer signs after three months, and over half remained cancer-free for at least a year. Common 

**Example with custom options**

In [21]:
# Summarize recent research on CAR-T therapy in India for leukemia patients.
# Give me latest updates about heart disease treatment in India.

response = client.chat.completions.create(
    model="gpt-4o-search-preview",
    messages=[
        {"role": "user", "content": "Summarize recent research on CAR-T therapy in India for leukemia patients."}
    ],
    web_search_options={
        "search_context_size": "high"
    }
)

print(response.choices[0].message.content)

Recent advancements in CAR-T cell therapy in India have significantly improved treatment options for leukemia patients. CAR-T cell therapy involves modifying a patient's T cells to target and destroy cancer cells, offering a personalized approach to cancer treatment.

**Development of Indigenous CAR-T Therapies**

In October 2023, ImmunoACT, an IIT Bombay-incubated company, received approval from the Central Drugs Standard Control Organization (CDSCO) for NexCAR19, India's first humanized CD19-targeted CAR-T cell therapy. This therapy is designed to treat relapsed or refractory B-cell lymphomas and leukemia. Clinical trials demonstrated an overall response rate of approximately 70%, with a favorable safety profile. Notably, NexCAR19 is priced between ₹30-40 lakh per patient, significantly lower than similar treatments available internationally, which can cost ₹3-4 crore. ([timesofindia.indiatimes.com](https://timesofindia.indiatimes.com/home/science/indias-1st-car-t-cell-therapy-develo

**Purpose of user_location**

This field allows web-enhanced GPT models (like gpt-4o-search-preview) to personalize search results based on your geographical context — useful for:

- Region-specific laws, treatments, or policies
- Local events, services, and prices
- Language/regional trends

In [11]:
def build_web_search_options(
    context_size: str = "medium",
    city: str     = None,
    region: str   = None,
    country: str  = None,
    timezone: str = None
) -> dict:
    """
    Builds a valid web_search_options dictionary for OpenAI web search.
    All location fields are optional, but country is recommended.

    context_size: one of "low", "medium", "high"
    """

    user_location = {
        "type": "approximate",
        "approximate": {}
    }

    if city:
        user_location["approximate"]["city"] = city
    if region:
        user_location["approximate"]["region"] = region
    if country:
        user_location["approximate"]["country"] = country
    if timezone:
        user_location["approximate"]["timezone"] = timezone

    return {
        "search_context_size": context_size,
        "user_location": user_location
    }

In [22]:
import json

options = build_web_search_options(
    context_size = "high",
    city         = "Bangalore",          # free text strings
    region       = "Karnataka",          # free text strings
    country      = "IN",                 # two-letter ISO country code, like US
    timezone     = "Asia/Kolkata"        # is an IANA timezone like America/Chicago
)

print(json.dumps(options, indent=2))

{
  "search_context_size": "high",
  "user_location": {
    "type": "approximate",
    "approximate": {
      "city": "Bangalore",
      "region": "Karnataka",
      "country": "IN",
      "timezone": "Asia/Kolkata"
    }
  }
}


In [23]:
response = client.chat.completions.create(
    model="gpt-4o-search-preview",
    #model="o4-mini",
    messages=[
        {"role": "user", "content": "Any new breakthroughs in diabetes treatment"}
    ],
    web_search_options=options
)

print(response.choices[0].message.content)

Recent advancements in diabetes treatment have introduced several promising therapies and technologies:

**1. FDA Approval of Ozempic for Kidney Disease Risk Reduction**

In January 2025, the U.S. Food and Drug Administration approved Novo Nordisk's Ozempic (semaglutide) to reduce the risk of kidney failure and disease progression, as well as death due to heart problems in diabetes patients with chronic kidney disease. This marks the first GLP-1 treatment option for individuals with type 2 diabetes and chronic kidney disease. ([reuters.com](https://www.reuters.com/business/healthcare-pharmaceuticals/us-fda-approves-novo-nordisks-diabetes-drug-reduce-risk-worsening-kidney-disease-2025-01-28/?utm_source=openai))

**2. Tirzepatide's Role in Preventing Type 2 Diabetes**

Eli Lilly's weight-loss drug, tirzepatide, demonstrated remarkable effectiveness in preventing diabetes, with nearly 99% of patients remaining diabetes-free after three years of treatment. The trial involved 1,032 adults w

No impact of location !!!

---
#### new-style tool calling format
---

the OpenAI /v1/responses endpoint, where you directly specify tools like:

- web_search_preview
- code_interpreter
- retrieval
- function_calling (via tool_choice)

You explicitly list tools in the request, and OpenAI decides how to use them based on your input.

In [24]:
response = client.responses.create(
    model ="gpt-4o",
    input ="What are the best restaurants around yelahanka?",
    tools =[
        {
            "type": "web_search_preview",
            "user_location": {
                "type": "approximate",
                "country": "IN",
                "city": "Bangalore",
                "region": "India"
            }
        }
    ]
)

print(response.output_text)

Yelahanka, a vibrant suburb in Bangalore, offers a diverse culinary scene catering to various tastes. Here are some top-rated restaurants in the area:

**[Nysa Sky Bar - Pubs in Yelahanka | Bangalore](https://nysaskybar.com/?utm_source=openai)**  
**Closed · 4.6 (534 reviews)**  
_5th Floor, 4H55+4P7 Clarion Hotel, 15, main road, Attur Layout, Yelahanka New Town, Kempanahalli, Bengaluru, Karnataka 560064, India_  
A rooftop bar known for its breathtaking views, live music, signature cocktails, and a fusion menu, making it ideal for brunches, date nights, and weekend gatherings.

**[The Druid Garden](https://www.thedruidgarden.in/?utm_source=openai)**  
**Closed · $$$ · 4.2 (9433 reviews)**  
_Century Corbel - Commercial, 40/1, Sahakar Nagar Main Rd, Park View Layout, Bengaluru, Karnataka 560092, India_  
Famous for its craft brews, wood-fired pizzas, and modern European-style cuisine, offering a haven for quality and innovation.

**[House Of Commons](https://houseofcommons.in/?utm_sour

> This forces the model to use the web_search_preview tool with the specified location.


**Example for a Healthcare Query**

In [25]:
response = client.responses.create(
    model="gpt-4o",
    input="What are the latest diabetes treatment options in India?",
    tools=[
        {
            "type": "web_search_preview",
            "user_location": {
                "type": "approximate",
                "country": "IN",
                "city": "Mumbai",
                "region": "Maharashtra",
                "timezone": "Asia/Kolkata"
            }
        }
    ]
)

print(response.output_text)


India has recently witnessed significant advancements in diabetes treatment, introducing innovative medications and technologies to enhance patient care.

**1. Introduction of GLP-1 Receptor Agonists:**

- **Mounjaro (Tirzepatide):** Eli Lilly launched Mounjaro in India in March 2025, following approval from the Drug Controller General of India. This once-weekly injectable drug aids in blood sugar regulation and weight management. The Mounjaro KwikPen, offering dosages from 2.5 mg to 15 mg, was introduced in August 2025, with prices starting at ₹14,000 for the 2.5 mg dose. ([reuters.com](https://www.reuters.com/business/healthcare-pharmaceuticals/eli-lilly-launches-weight-loss-drug-mounjaro-india-after-drug-regulator-approval-2025-03-20/?utm_source=openai))

- **Wegovy (Semaglutide):** Novo Nordisk's Wegovy, another GLP-1 receptor agonist, has been approved in India for weight management and type 2 diabetes treatment. Its availability provides patients with additional therapeutic optio

#### comparison with Old Format
    
| Feature                       | `chat.completions.create(...)`      | `responses.create(...)` (✅ new-style)        |
| ----------------------------- | ----------------------------------- | -------------------------------------------- |
| Endpoint                      | `/v1/chat/completions`              | `/v1/responses`                              |
| Tools specified?              | ❌ Indirect via `web_search_options` | ✅ Explicit via `tools=[...]`                 |
| Location-sensitive Web Search | ❌ Weak, sometimes ignored           | ✅ Strong control via `user_location`         |
| Multi-tool behavior           | ❌ No                                | ✅ Supports multiple tools (e.g., code + web) |


---
#### search_context_size in web_search_preview

**What it controls:**

How much text (context) is retrieved from the web to help generate the LLM response.

---

| Setting              | Context Detail        | Cost Impact        | Quality               | Latency     | Use Case Example                   |
| -------------------- | --------------------- | ------------------ | --------------------- | ----------- | ---------------------------------- |
| `"low"`              | Snippets or headlines | Cheapest         | ❗ Might miss nuance   | 🚀 Fastest  | Real-time headlines, location tips |
| `"medium"` (default) | Balanced context      | Moderate        |  Good for general Qs |  Balanced | Health/tech news, product queries  |
| `"high"`             | Full paragraphs/pages | Possibly billed |  Best for deep Qs   |  Slowest  | Medical advances, legal analysis   |


In [16]:
tools = [{
    "type": "web_search_preview",
    "user_location": {
        "type": "approximate",
        "city": "Delhi",
        "region": "Delhi",
        "country": "IN"
    },
    "search_context_size": "low"
}]

In [17]:
response = client.responses.create(
    model="gpt-4o",
    input="Are there any new clinical trials on reversing diabetes in India?",
    tools=tools
)

print(response.output_text)

Yes, there are several recent clinical trials and initiatives in India focused on reversing diabetes:

1. **Diabetes Remission in India (DiRemI) Study**: This prospective, open-label, matched control trial aims to evaluate the effectiveness of a lifestyle intervention program in achieving diabetes remission. Conducted at the Freedom from Diabetes Clinic, the study involves participants from Pune and Ahmedabad and is registered with the Clinical Trials Registry-India (CTRI/2023/06/053885). ([pmc.ncbi.nlm.nih.gov](https://pmc.ncbi.nlm.nih.gov/articles/PMC11213318/?utm_source=openai))

2. **Redial Clinic's Holistic Diabetes Reversal Initiative**: Launched in June 2025 in Delhi NCR, this nationwide program targets reversing type 2 diabetes, hypertension, and obesity through clinical lifestyle interventions. The initiative aims to impact over 100,000 patients by 2026, integrating physician-led care with customized nutrition, resistance training, and digital habit coaching. ([business-standa